# Exercises - using your package

### 1. Something familiar

> **TASK 1:** import your package and use it to solve exercise 1 from Part 1 (Canon ball range). Compute the range distributions for the two parameter sets and plot their histograms.

In [ ]:
# import packages ...

# FOR  TAs
import numpy as np 
from my_package import propagate

In [9]:
# your solution ...

In [ ]:
# FOR TAs

# Exercise 1

# seed the random number generator for reproducibility
rng = np.random.default_rng(42) 
n_samples = 10_000
rad_per_deg = np.pi / 180    

# define how the range depends on angle, speed, and gravity
def get_range(theta: float, v: float, g: float) -> float:
    return v**2 / g * np.sin(2 * theta) 

# define input kinds and parameters in the same order as expected in get_range()
inputs = [
    ("normal", 35 * rad_per_deg, 1 * rad_per_deg), # theta
    ("normal", 10, 0.1), # v
    ("constant", 9.82) # g
]

# propagate the inputs to get the range distribution
ranges = propagate(rng, get_range, inputs, n_samples=n_samples)

# compute mean and standard deviation of the range
mean_range = np.mean(ranges)
std_range = np.std(ranges)

# print output
print(f"The estimated range is {mean_range:.2f} +- {std_range:.2f} m")

The estimated range is 9.57 +- 0.23 m


### 2. Can we trust our detector?

A particle detector records discrete counts during a fixed observation interval. Even when no signal is present, background radiation produces counts.

Suppose background counts are Poisson distributed with mean 2, while a real event contributes an additional independent Poisson signal with mean 6.

The detector announces a discovery whenever the total observed count reaches a threshold $k$.

**Question:** *what threshold $k$ should we choose to avoid false alarms without missing too many real events?*

The statistical model is:

$$
N_{background} \sim \text{Poisson}(\lambda=2) \\

N_{events} \sim \text{Poisson}(\lambda=6)
$$

Where the detector reports $N = N_{background} + N_{events}$, and the detection function is:

$$
D(N) = \begin{cases}
1 & \text{if } N \geq k \\
0 & \text{if } N < k
\end{cases}
$$

> **TASK 2**: Estimate the probability of detection for thresholds $k=3,4,5,6,7$

In [22]:
# your solution ...

# FOR TAs
rng = np.random.default_rng(42)

def detection(n_background: int, n_events: int, k: float) -> int:
    return int(n_background + n_events >= k)

inputs = [
    ("poisson", 2),
    ("poisson", 6),
    ("constant", 3)
]

detections = propagate(rng, detection, inputs, n_samples=1_000)

print(f"The probability of detection for k = 3 is {np.mean(detections):.2f}")

for k in (4, 5, 6, 7):
    inputs[-1] = ("constant", k)
    detections = propagate(rng, detection, inputs, n_samples=1_000)
    print(f"The probability of detection for k = {k} is {np.mean(detections):.2f}")

The probability of detection for k = 3 is 0.98
The probability of detection for k = 4 is 0.95
The probability of detection for k = 5 is 0.90
The probability of detection for k = 6 is 0.80
The probability of detection for k = 7 is 0.70


> **TASK 3:** Repeat with no signal / reel events present (replacing events as a `constant` type zero input) to estimate the "false-alarm probability"

In [27]:
# your solution ...

# FOR TAs
rng = np.random.default_rng(42)

inputs = [
    ("poisson", 2),
    ("constant", 0),
    ("constant", 3)
]

for k in (3, 4, 5, 6, 7):
    inputs[-1] = ("constant", k)
    detections = propagate(rng, detection, inputs, n_samples=1_000)
    print(f"The false-alarm probability for k = {k} is {np.mean(detections):.2f}")


The false-alarm probability for k = 3 is 0.32
The false-alarm probability for k = 4 is 0.13
The false-alarm probability for k = 5 is 0.06
The false-alarm probability for k = 6 is 0.02
The false-alarm probability for k = 7 is 0.01


> **TASK 4:** Recommend a threshold that keeps the false-alarm probability below 5% while detecting as many reel events as possible

your answer ...

FOR TAs

Choosing $k=6$ gives an estimated false-alaram probability of 2% while triggering detection 80% of the time, i.e. 20% of measurements are rejected. 

Choosing $k=5$ lowers that fraction to 10% at the cost of a 6% change of a false alarm.

> **TASK 5:** Explain why the mean of the simulated $D$ outputs is more useful than reporting a conventional propagated error bar

FOR TAs

The mean of the outputs is a probability of detection derived from the statistical Poissonian model. The distribution of outputs is a bi-modal pile of binary 0/1 observations. Talking about a +- errorbar in the Gaussian sense is not sensical. 

The probability has a theoretical value determined by the $\lambda$ and $k$ parameters and is as such not "uncertain". We could however estimate the variability in our finite sample estimation by repeating over multiple seeds.


**Expected results for two thresholds**

| value | false-alarm | detection probability |
| --- | --- | --- |
|$k=5$ |	5.27% |	90.04% |
|$k=6$ |	1.66% |	80.88% |

**What they learn:** Monte Carlo can simulate decision-making and competing risks, not just propagate numerical uncertainty. The output of interest is a probability, and a statistically informed decision requires considering competing objectives.

### 3. Which experimental method is better?

...


### 4. How many particles hit our detector?

...